In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf transformers accelerate sentence-transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

os.makedirs("./docs", exist_ok=True)

In [ ]:
import shutil

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        shutil.move(
            filename,
            os.path.join("./docs", filename)
        )

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_directory = "./docs"

documents = []

file_paths = [
    os.path.join(pdf_directory, file)
    for file in os.listdir(pdf_directory)
    if file.lower().endswith(".pdf")
]

for file_path in file_paths:

    loader = PyPDFLoader(file_path)

    documents.extend(loader.load())

print("Number of pages loaded:", len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

text_splitted_document = text_splitter.split_documents(documents)

print("Number of chunks:", len(text_splitted_document))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    text_splitted_document,
    embeddings
)

print("FAISS vector store created successfully!")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [ ]:
evaluation_dataset = [

    # --------------------------------------------------
    # DIRECT FACTUAL QUESTIONS
    # --------------------------------------------------

    {
        "question": "What is the remaining mortgage balance?",
        "expected": "342,600",
        "category": "Direct"
    },

    {
        "question": "What is the monthly mortgage payment?",
        "expected": "2,180",
        "category": "Direct"
    },

    {
        "question": "What is the interest rate on the mortgage?",
        "expected": "5.35",
        "category": "Direct"
    },

    {
        "question": "How much remains on the car loan?",
        "expected": "11,400",
        "category": "Direct"
    },

    {
        "question": "What is the monthly car loan payment?",
        "expected": "365",
        "category": "Direct"
    },

    {
        "question": "How much is currently in the emergency fund?",
        "expected": "15,000",
        "category": "Direct"
    },

    {
        "question": "What is the target for the emergency fund?",
        "expected": "24,000",
        "category": "Direct"
    },

    {
        "question": "How much is currently in the brokerage account?",
        "expected": "58,300",
        "category": "Direct"
    },


    # --------------------------------------------------
    # PARAPHRASED QUESTIONS
    # --------------------------------------------------

    {
        "question": "How much does the household pay every month toward the home loan?",
        "expected": "2,180",
        "category": "Paraphrased"
    },

    {
        "question": "What amount has been set aside as the household's financial safety cushion?",
        "expected": "15,000",
        "category": "Paraphrased"
    },

    {
        "question": "What sum has been accumulated for repairing the house?",
        "expected": "6,200",
        "category": "Paraphrased"
    },

    {
        "question": "How much money has been put away for the eventual replacement of Priya's car?",
        "expected": "4,800",
        "category": "Paraphrased"
    },

    {
        "question": "What is the current value of the household's taxable investment account?",
        "expected": "58,300",
        "category": "Paraphrased"
    },


    # --------------------------------------------------
    # CONCEPTUAL QUESTIONS
    # --------------------------------------------------

    {
        "question": "Why does the household keep several savings accounts instead of putting all the money into one account?",
        "expected": "separate goals",
        "category": "Conceptual"
    },

    {
        "question": "Why is the emergency fund kept separate from the home repair fund?",
        "expected": "emergencies",
        "category": "Conceptual"
    },

    {
        "question": "Why do Daniel and Priya consider the car loan more urgent than the mortgage?",
        "expected": "higher rate",
        "category": "Conceptual"
    },

    {
        "question": "Why don't they consider their brokerage account to be emergency savings?",
        "expected": "market",
        "category": "Conceptual"
    },

    {
        "question": "Why is the vacation fund considered more flexible than the vehicle fund?",
        "expected": "vacation can be postponed",
        "category": "Conceptual"
    },


    # --------------------------------------------------
    # COMPARISON QUESTIONS
    # --------------------------------------------------

    {
        "question": "Which has the higher interest rate, the mortgage or the car loan?",
        "expected": "car loan",
        "category": "Comparison"
    },

    {
        "question": "How does the purpose of the emergency fund differ from the home repair fund?",
        "expected": "emergency fund",
        "category": "Comparison"
    },

    {
        "question": "How are the brokerage account and cryptocurrency holdings treated differently?",
        "expected": "cryptocurrency",
        "category": "Comparison"
    },

    {
        "question": "Which is considered more urgent, replacing the car or taking the vacation?",
        "expected": "vehicle",
        "category": "Comparison"
    },


    # --------------------------------------------------
    # MULTI-HOP QUESTIONS
    # --------------------------------------------------

    {
        "question": "Why do they prioritize paying off the car loan before making extra mortgage payments?",
        "expected": "car loan",
        "category": "Multi-hop"
    },

    {
        "question": "What financial priority comes after finishing the car loan?",
        "expected": "emergency fund",
        "category": "Multi-hop"
    },

    {
        "question": "Why does the household continue retirement contributions even while dealing with shorter-term goals?",
        "expected": "long-term",
        "category": "Multi-hop"
    },


    # --------------------------------------------------
    # MISSING INFORMATION / HALLUCINATION TESTS
    # --------------------------------------------------

    {
        "question": "What is the name of the institution that currently services the mortgage?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What is the current balance of the college fund?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What specific car model does Priya plan to buy?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What is their planned vacation destination?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What medications are Daniel and Priya currently taking?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },


    # --------------------------------------------------
    # NUMERICAL / CALCULATION QUESTIONS
    # --------------------------------------------------

    {
        "question": "How much more money is needed to reach the emergency fund target?",
        "expected": "9,000",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed to reach the home repair fund target?",
        "expected": "3,800",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed to reach the vacation fund target?",
        "expected": "1,850",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed for the vehicle savings account to reach its target?",
        "expected": "7,200",
        "category": "Calculation"
    }
]

In [ ]:
for i, test in enumerate(evaluation_dataset):

    query = test["question"]

    result = retriever.invoke(query)

    print("\n" + "=" * 70)
    print(f"QUESTION {i + 1}")
    print("=" * 70)

    print("Question:", query)
    print("Category:", test["category"])

    print("\nRetrieved Chunks:")

    for j, doc in enumerate(result):
        print(f"\n--- Chunk {j + 1} ---")
        print(doc.page_content)

In [ ]:
from transformers import pipeline
import torch

llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_new_tokens=150,
)

In [ ]:
def ask_rag(question, context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise question-answering assistant. "
                "Answer ONLY using the information in the provided context. "
                "If the answer is not in the context, respond exactly with: "
                "\"I couldn't find this information.\" "
                "Do not guess, infer, or add any detail not explicitly stated in the context. "
                "Be concise and directly answer what is asked."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    output = llm_pipeline(
        messages,
        max_new_tokens=150,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    return output[0]["generated_text"][-1]["content"]

In [ ]:
import time

def run_rag_evaluation(evaluation_dataset, retriever, ask_rag_fn, verbose=True):
    """
    evaluation_dataset : list of dicts, each with "question", "expected", "category"
    retriever          : function(question) -> context string (your existing retrieval step)
    ask_rag_fn         : the ask_rag(question, context) function defined earlier
    """
    results = []

    for i, item in enumerate(evaluation_dataset, start=1):
        question = item["question"]
        expected = item.get("expected", "N/A")
        category = item.get("category", "Uncategorized")


        context = retriever(question)


        start = time.time()
        answer = ask_rag_fn(question, context)
        elapsed = time.time() - start

        result = {
            "index": i,
            "category": category,
            "question": question,
            "expected": expected,
            "context": context,
            "model_answer": answer,
            "time_sec": round(elapsed, 2),
        }
        results.append(result)

        if verbose:
            print("=" * 70)
            print(f"QUESTION {i} [{category}]")
            print("=" * 70)
            print(f"Question: {question}")
            print(f"Expected: {expected}")
            print(f"Model Answer: {answer}")
            print(f"Time: {elapsed:.2f}s")
            print()

    return results


def summarize_results(results, save_csv_path=None):
    """
    Prints a per-category and overall breakdown.
    Does NOT auto-grade (substring matching is unreliable, as we found) —
    it just organizes results so you can eyeball each one against the source doc.
    """
    from collections import defaultdict

    by_category = defaultdict(list)
    for r in results:
        by_category[r["category"]].append(r)

    print("\n" + "=" * 70)
    print("SUMMARY BY CATEGORY")
    print("=" * 70)
    for category, items in by_category.items():
        print(f"\n{category} ({len(items)} questions):")
        for r in items:
            print(f"  Q{r['index']}: {r['question'][:60]}...")
            print(f"    -> {r['model_answer'][:100]}")

    if save_csv_path:
        import csv
        with open(save_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["index", "category", "question", "expected", "model_answer", "time_sec"])
            writer.writeheader()
            for r in results:
                writer.writerow({k: r[k] for k in writer.fieldnames})
        print(f"\nSaved results to {save_csv_path}")

    return by_category




In [ ]:
def get_context(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return context


In [ ]:
results = run_rag_evaluation(
    evaluation_dataset=evaluation_dataset,
    retriever=get_context,
    ask_rag_fn=ask_rag,
    verbose=True
)

summarize_results(results, save_csv_path="rag_results.csv")

# **Evaluation Harness**

Performing three different kinds of chunking to get the best accuracy results

In [ ]:
!pip install -q langchain langchain-community langchain-experimental langchain-text-splitters sentence-transformers faiss-cpu pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter


loader = PyPDFLoader("/content/docs/household_notes.pdf")
raw_documents = loader.load()


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Fixed Chunking

In [ ]:
def build_fixed_chunking_retriever(chunk_size=500, k=4):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Fixed Chunking] Created {len(chunks)} chunks (size={chunk_size}, overlap=0)")
    return retriever


fixed_retriever = build_fixed_chunking_retriever(chunk_size=500, k=4)

# Overlap Chunking

In [ ]:
def build_overlap_chunking_retriever(chunk_size=500, chunk_overlap=100, k=4):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Overlap Chunking] Created {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")
    return retriever


overlap_retriever = build_overlap_chunking_retriever(chunk_size=500, chunk_overlap=100, k=4)

# Semantic Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

def build_semantic_chunking_retriever(breakpoint_threshold_amount=80, k=4):
    semantic_splitter = SemanticChunker(
        embedding_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=breakpoint_threshold_amount,
    )
    chunks = semantic_splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Semantic Chunking] Created {len(chunks)} chunks")
    return retriever


semantic_retriever = build_semantic_chunking_retriever(breakpoint_threshold_amount=80, k=4)

In [ ]:
def get_context_fn(retriever_obj):
    """Wraps any LangChain retriever into the get_context(question) function
    that run_rag_evaluation expects."""
    def get_context(question):
        docs = retriever_obj.invoke(question)
        return "\n\n".join([doc.page_content for doc in docs])
    return get_context


strategies = {
    "fixed": fixed_retriever,
    "overlap": overlap_retriever,
    "semantic": semantic_retriever,
}

all_results = {}

for name, retriever_obj in strategies.items():
    print(f"\n{'#'*70}\nRunning evaluation with: {name.upper()} chunking\n{'#'*70}")
    results = run_rag_evaluation(
        evaluation_dataset=evaluation_dataset,
        retriever=get_context_fn(retriever_obj),
        ask_rag_fn=ask_rag,
        verbose=False,
    )
    summarize_results(results, save_csv_path=f"rag_results_{name}.csv")
    all_results[name] = results